In [1]:
import re
import itertools

class Expr:
    """Represents a node in the propositional logic expression tree."""
    def __init__(self, op, *args):
        self.op = op
        self.args = args

    def evaluate(self, model):
        """Recursively evaluates the truth value of the expression."""
        if not self.args:
            return model[self.op]

        args_eval = [arg.evaluate(model) for arg in self.args]
        if self.op == '~':    return not args_eval[0]
        if self.op == '&':    return args_eval[0] and args_eval[1]
        if self.op == '|':    return args_eval[0] or args_eval[1]
        if self.op == '>>':   return (not args_eval[0]) or args_eval[1]
        if self.op == '<<>>': return args_eval[0] == args_eval[1]
        raise ValueError(f"Unknown operator: {self.op}")

    def get_symbols(self):
        """Extracts unique variable names from the expression tree."""
        if not self.args:
            return {self.op}
        return set().union(*(arg.get_symbols() for arg in self.args))


def parse_expr(expr_str):
    """Parses a string logic formula into an Expr tree using Python's eval."""
    s = expr_str.strip().replace('<<>>', '==').replace('>>', '<=')
    tokens = re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*', s)
    if not tokens:
        raise ValueError(f"No symbols found in: '{expr_str}'")

    namespace = {t: Expr(t) for t in tokens}

    class LogicWrapper:
        def __init__(self, obj): self.obj = obj
        def __invert__(self): return LogicWrapper(Expr('~', self.obj))
        def __or__(self, other): return LogicWrapper(Expr('|', self.obj, other.obj))
        def __and__(self, other): return LogicWrapper(Expr('&', self.obj, other.obj))
        def __le__(self, other): return LogicWrapper(Expr('>>', self.obj, other.obj))
        def __eq__(self, other): return LogicWrapper(Expr('<<>>', self.obj, other.obj))

    wrapped_ns = {k: LogicWrapper(v) for k, v in namespace.items()}
    s_wrapped = re.sub(r'([a-zA-Z_][a-zA-Z0-9_]*)', r'wrapped_ns["\1"]', s)
    return eval(s_wrapped, {"wrapped_ns": wrapped_ns}).obj


def TT_ENTAILS(KB_strings, alpha_string):
    """Generates the truth table and checks for KB ⊨ alpha entailment."""
    KB_exprs = [parse_expr(s) for s in KB_strings]
    alpha_expr = parse_expr(alpha_string)

    # Collect and sort all unique symbols
    symbols = set()
    for expr in KB_exprs:
      symbols.update(expr.get_symbols())
    symbols.update(alpha_expr.get_symbols())
    symbols = sorted(list(symbols))

    # Print clean table header
    headers = symbols + ["KB", "alpha"]
    fmt = "".join([f"{{:<{max(len(h)+2, 8)}}}" for h in headers])
    print("\n" + fmt.format(*headers))
    print("-" * (len(headers) * 8))

    entails = True

    # Enumerate all possible truth assignments (False first to match lecturer slide format)
    for values in itertools.product([False, True], repeat=len(symbols)):
        model = dict(zip(symbols, values))

        kb_val = all(expr.evaluate(model) for expr in KB_exprs)
        alpha_val = alpha_expr.evaluate(model)

        # Print truth table row
        row_vals = [str(model[sym]).lower() for sym in symbols]
        row_vals.extend([str(kb_val).lower(), str(alpha_val).lower()])
        print(fmt.format(*row_vals))

        # Check for model counterexample: KB is True, alpha is False
        if kb_val and not alpha_val:
            entails = False

    print("-" * (len(headers) * 8))
    print(f"KB ⊨ alpha -> {entails}\n")
    return entails


# --- Interactive Input Flow ---
if __name__ == "__main__":
    print("Allowed Operators: ~ (NOT), & (AND), | (OR), >> (Implication), <<>> (Biconditional)")
    print("Enter KB sentences line by line. Press [Enter] on an empty line when done.")
    print("-" * 60)

    user_kb = []
    while True:
        sentence = input(f"Enter KB sentence #{len(user_kb) + 1}: ").strip()
        if not sentence:
            break
        user_kb.append(sentence)

    if not user_kb:
        print("Error: Knowledge base cannot be empty.")
        exit()

    user_query = input("Enter the query sentence (alpha): ").strip()

    try:
        TT_ENTAILS(user_kb, user_query)
    except Exception as e:
        print(f"\nParsing Error: {e}")

Allowed Operators: ~ (NOT), & (AND), | (OR), >> (Implication), <<>> (Biconditional)
Enter KB sentences line by line. Press [Enter] on an empty line when done.
------------------------------------------------------------
Enter KB sentence #1: (A | C) & (B | ~C)
Enter KB sentence #2: 
Enter the query sentence (alpha): (A | B)

A       B       C       KB      alpha   
----------------------------------------
false   false   false   false   false   
false   false   true    false   false   
false   true    false   false   true    
false   true    true    true    true    
true    false   false   true    true    
true    false   true    false   true    
true    true    false   true    true    
true    true    true    true    true    
----------------------------------------
KB ⊨ alpha -> True

